In [1]:
# ============================================================
# Project: EuroTrans Analytics
# Notebook: 03_Gold_Delivery_Analytics
# Layer: Gold
#
# Description:
# Build the Gold Fact Shipment table for the semantic model.
# ============================================================

# ------------------------------------------------------------
# Import Libraries
# ------------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    round,
    current_timestamp
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

# ------------------------------------------------------------
# Create Spark Session
# ------------------------------------------------------------

spark = SparkSession.builder.getOrCreate()

# ------------------------------------------------------------
# ETL Audit
# ------------------------------------------------------------

audit_log = []

print("=" * 60)
print("Loading Silver tables...")
print("=" * 60)

# ------------------------------------------------------------
# Load Silver Tables
# ------------------------------------------------------------

shipment = spark.table("silver_shipment")
customer = spark.table("silver_customer")
product = spark.table("silver_product")
carrier = spark.table("silver_carrier")
route = spark.table("silver_route")

rows_before = shipment.count()

# ------------------------------------------------------------
# Create Gold Dataset
# ------------------------------------------------------------

gold = (
    shipment
        .join(customer, "CustomerID", "left")
        .join(product, "ProductID", "left")
        .join(carrier, "CarrierID", "left")
        .join(route, "RouteID", "left")
)

# ------------------------------------------------------------
# Business Metrics
# ------------------------------------------------------------

gold = gold.withColumn(
    "ProfitMarginPct",
    when(
        col("RevenueEUR") > 0,
        round(
            col("ProfitEUR") / col("RevenueEUR"),
            4
        )
    ).otherwise(0)
)

gold = gold.withColumn(
    "RevenuePerKg",
    when(
        col("WeightKg") > 0,
        round(
            col("RevenueEUR") / col("WeightKg"),
            2
        )
    ).otherwise(0)
)

gold = gold.withColumn(
    "CostPerKg",
    when(
        col("WeightKg") > 0,
        round(
            col("TotalTransportCostEUR") / col("WeightKg"),
            2
        )
    ).otherwise(0)
)

gold = gold.withColumn(
    "IsDelayed",
    when(
        col("DelayHours") > 0,
        "Yes"
    ).otherwise("No")
)

# ------------------------------------------------------------
# Select Business Columns
# ------------------------------------------------------------

gold_fact = gold.select(

    # Keys
    "ShipmentID",
    "ShipmentDate",
    "CustomerID",
    "ProductID",
    "CarrierID",
    "RouteID",

    # Measures
    "WeightKg",
    "RevenueEUR",

    "FuelCostEUR",
    "TollCostEUR",
    "DriverCostEUR",
    "MaintenanceCostEUR",
    "HandlingCostEUR",

    "TotalTransportCostEUR",
    "ProfitEUR",
    "ProfitMarginPct",

    # KPIs
    "RevenuePerKg",
    "CostPerKg",

    "DeliveryStatus",
    "DelayHours",
    "IsDelayed"
)

rows_after = gold_fact.count()
duplicates_removed = 0

# ------------------------------------------------------------
# Save Gold Fact Table
# ------------------------------------------------------------

(
    gold_fact.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_fact_shipments")
)

audit_log.append((
    "03_Gold_Delivery_Analytics",
    "Gold",
    "gold_fact_shipments",
    rows_before,
    rows_after,
    duplicates_removed,
    "Success"
))

# ------------------------------------------------------------
# Write ETL Audit Log
# ------------------------------------------------------------

audit_schema = StructType([
    StructField("Notebook", StringType(), False),
    StructField("Layer", StringType(), False),
    StructField("TableName", StringType(), False),
    StructField("RowsBefore", LongType(), False),
    StructField("RowsAfter", LongType(), False),
    StructField("DuplicatesRemoved", LongType(), False),
    StructField("Status", StringType(), False)
])

audit_df = spark.createDataFrame(audit_log, audit_schema)

audit_df = audit_df.withColumn(
    "RunTimestamp",
    current_timestamp()
)

(
    audit_df.write
        .mode("append")
        .format("delta")
        .saveAsTable("etl_audit_log")
)

print("\nETL audit successfully written.")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 60)
print("Gold Fact Shipments created successfully.")
print("=" * 60)

print(f"Rows: {rows_after}")
print(f"Columns: {len(gold_fact.columns)}")

display(gold_fact.limit(10))

StatementMeta(, b66e2549-bfe2-47d0-9f99-da0cbb6ab034, 3, Finished, Available, Finished, False)

Loading Silver tables...

ETL audit successfully written.
Gold Fact Shipments created successfully.
Rows: 50000
Columns: 21


SynapseWidget(Synapse.DataFrame, 275ed158-4340-48a5-bae4-a8b9c76dde77)